In [1]:
import pandas as pd

# 1. 엑셀 파일 경로
file_path = r"C:\Users\wngus\Documents\SK하이닉스_뉴스_자료_영문컬럼.xlsx"

# 2. 엑셀 읽기
df = pd.read_excel(file_path)

# 3. 기존 컬럼 확인
print("원본 컬럼명:", df.columns.tolist())

# 4. 컬럼명 매핑 (엑셀 → DB)
df = df.rename(columns={
    'news_date': 'published_date',
    'news_time': 'published_time',
    'change_rate': 'sentiment_score',
    'direction': 'sentiment',
    'title': 'title',
    'url': 'url',
    'source_domain': 'publisher_name'
})

# 5. published_at (timestamp) 생성: 날짜 + 시간 합치기
df['published_at'] = pd.to_datetime(df['published_date'].astype(str) + ' ' + df['published_time'].astype(str))

# 6. sentiment 기호로 변환
df['sentiment'] = df['sentiment'].map({
    '상승': '+',
    '하락': '-',
    '보합': 'NE'  # 중립(optional)
})

# 7. ticker, price_date 고정값 생성
df['ticker'] = '000660'
df['price_date'] = pd.to_datetime(df['published_date']).dt.date  # date 형식

# 8. 불필요 컬럼 정리
df_final = df[['ticker', 'published_at', 'publisher_name', 'title', 'url', 
               'sentiment', 'sentiment_score', 'price_date']]

# 9. 결과 미리 보기
print(df_final.head())

원본 컬럼명: ['news_date', 'news_time', 'change_rate', 'direction', 'title', 'url', 'source_domain']
   ticker        published_at publisher_name  \
0  000660 2025-06-23 17:26:00           fetv   
1  000660 2025-06-23 17:26:00           fetv   
2  000660 2025-06-23 17:26:00           fetv   
3  000660 2025-06-23 17:26:00           fetv   
4  000660 2025-06-23 17:26:00           fetv   

                              title  \
0  HL D&I한라, 이천 '부발역 에피트 에디션' 7월 분양   
1  HL D&I한라, 이천 '부발역 에피트 에디션' 7월 분양   
2  HL D&I한라, 이천 '부발역 에피트 에디션' 7월 분양   
3  HL D&I한라, 이천 '부발역 에피트 에디션' 7월 분양   
4  HL D&I한라, 이천 '부발역 에피트 에디션' 7월 분양   

                                                 url sentiment  \
0  https://www.fetv.co.kr/news/article.html?no=19...         +   
1  https://www.fetv.co.kr/news/article.html?no=19...         -   
2  https://www.fetv.co.kr/news/article.html?no=19...         -   
3  https://www.fetv.co.kr/news/article.html?no=19...         -   
4  https://www.fetv.co.kr/news/article.html?no=19.

In [2]:
save_path = r"C:\Users\wngus\Documents\SK하이닉스_정리본.csv"
df_final.to_csv(save_path, index=False, encoding='utf-8-sig')

In [4]:
import os
from dotenv import load_dotenv
import psycopg2

# .env 파일 불러오기
load_dotenv(dotenv_path='stock.env')  # 경로는 필요시 수정

# 환경변수로부터 DB 정보 읽기
conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    port=os.getenv("DB_PORT")
)

cur = conn.cursor()

cur.execute("""
    SELECT *
    FROM stock_price
    WHERE ticker = '000660'
    ORDER BY price_date DESC
""")
rows = cur.fetchall()
for row in rows:
    print(row)

conn.close()

In [15]:
import pandas as pd
import psycopg2
import os
from dotenv import load_dotenv

# 환경변수 로드
load_dotenv(dotenv_path='stock.env')

# 엑셀 파일 경로
file_path = r"C:\Users\wngus\Documents\SK하이닉스_정리본.xlsx"

# 엑셀 데이터 로드
df = pd.read_excel(file_path)

# DB 연결
conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    port=os.getenv("DB_PORT")
)
cur = conn.cursor()

for _, row in df.iterrows():
    # 1. 발행처 처리 (UNIQUE 제약 조건 있으므로 안전)
    try:
        cur.execute("""
            INSERT INTO publisher (name) 
            VALUES (%s) 
            ON CONFLICT (name) DO NOTHING
            RETURNING publisher_id
        """, (row['publisher_name'],))
        
        if cur.rowcount > 0:
            publisher_id = cur.fetchone()[0]
        else:
            cur.execute("SELECT publisher_id FROM publisher WHERE name = %s", (row['publisher_name'],))
            publisher_id = cur.fetchone()[0]
    except Exception as e:
        print(f"발행처 처리 오류: {e}")
        continue

    # 2. 뉴스 본문 저장
    try:
        cur.execute("""
            INSERT INTO news (ticker, publisher_id, title, url, published_at, price_date)
            VALUES (%s, %s, %s, %s, %s, %s)
            ON CONFLICT (url) DO NOTHING
            RETURNING news_id
        """, (
            '000660', 
            publisher_id,
            row['title'],
            row['url'],
            row['published_at'],
            row['price_date']
        ))
        
        if cur.rowcount > 0:
            news_id = cur.fetchone()[0]
        else:
            cur.execute("SELECT news_id FROM news WHERE url = %s", (row['url'],))
            news_id = cur.fetchone()[0]
    except Exception as e:
        print(f"뉴스 저장 오류: {e}")
        continue

    # 3. 감성 분석 결과 저장
    try:
        cur.execute("""
            INSERT INTO news_sentiment (news_id, sentiment, sentiment_score)
            VALUES (%s, %s, %s)
        """, (
            news_id,
            row['sentiment'],
            float(row['sentiment_score'])
        ))
    except Exception as e:
        print(f"감성 분석 결과 저장 오류: {e}")

conn.commit()


뉴스 저장 오류: 오류:  ON CONFLICT 절을 사용하는 경우, unique 나 exclude 제약 조건이 있어야 함

발행처 처리 오류: 오류:  현재 트랜잭션은 중지되어 있습니다. 이 트랜잭션을 종료하기 전까지는 모든 명령이 무시될 것입니다

발행처 처리 오류: 오류:  현재 트랜잭션은 중지되어 있습니다. 이 트랜잭션을 종료하기 전까지는 모든 명령이 무시될 것입니다

발행처 처리 오류: 오류:  현재 트랜잭션은 중지되어 있습니다. 이 트랜잭션을 종료하기 전까지는 모든 명령이 무시될 것입니다

발행처 처리 오류: 오류:  현재 트랜잭션은 중지되어 있습니다. 이 트랜잭션을 종료하기 전까지는 모든 명령이 무시될 것입니다

발행처 처리 오류: 오류:  현재 트랜잭션은 중지되어 있습니다. 이 트랜잭션을 종료하기 전까지는 모든 명령이 무시될 것입니다

발행처 처리 오류: 오류:  현재 트랜잭션은 중지되어 있습니다. 이 트랜잭션을 종료하기 전까지는 모든 명령이 무시될 것입니다

발행처 처리 오류: 오류:  현재 트랜잭션은 중지되어 있습니다. 이 트랜잭션을 종료하기 전까지는 모든 명령이 무시될 것입니다

발행처 처리 오류: 오류:  현재 트랜잭션은 중지되어 있습니다. 이 트랜잭션을 종료하기 전까지는 모든 명령이 무시될 것입니다

발행처 처리 오류: 오류:  현재 트랜잭션은 중지되어 있습니다. 이 트랜잭션을 종료하기 전까지는 모든 명령이 무시될 것입니다

발행처 처리 오류: 오류:  현재 트랜잭션은 중지되어 있습니다. 이 트랜잭션을 종료하기 전까지는 모든 명령이 무시될 것입니다

발행처 처리 오류: 오류:  현재 트랜잭션은 중지되어 있습니다. 이 트랜잭션을 종료하기 전까지는 모든 명령이 무시될 것입니다

발행처 처리 오류: 오류:  현재 트랜잭션은 중지되어 있습니다. 이 트랜잭션을 종료하기 전까지는 모든 명령이 무시될 것입니다

발행처 처리 오류: 오류:  현재 트랜잭션은 중지되어 있습니다. 이 트랜잭션을 종료하기 전까지는 모든 명령이 무시될 것입니다

발행처 처리 

In [7]:
query = """
SELECT news_id, title, summary, price_date
FROM news
WHERE ticker = '000660'
AND price_date = '2025-06-23';
"""

cur.execute(query)
rows = cur.fetchall()

for row in rows:
    print(row)

InterfaceError: cursor already closed

In [8]:
import pandas as pd

df = pd.read_sql_query(query, conn)
print(df.head())

C:\Users\wngus\AppData\Local\Temp\ipykernel_9188\1712246602.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


InterfaceError: connection already closed

In [21]:
import pandas as pd
import psycopg2
import os
from dotenv import load_dotenv

# 🔐 환경 변수 로드
load_dotenv('stock.env')

# 🧠 DB 연결
conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    port=os.getenv("DB_PORT")
)

inserted_count = 0

try:
    cur = conn.cursor()

    # 📥 엑셀 데이터 로드
    df = pd.read_excel(r"C:\Users\wngus\Documents\SK하이닉스_정리본.xlsx")
    df['published_at'] = pd.to_datetime(df['published_at'])
    df['price_date'] = pd.to_datetime(df['price_date'])
    df['ticker'] = df['ticker'].astype(str).str.zfill(6)  # 숫자로 읽힌 ticker 보정

    for _, row in df.iterrows():
        try:
            # ✅ ticker 존재 확인
            cur.execute("SELECT ticker FROM ticker WHERE ticker = %s", (row['ticker'],))
            if cur.fetchone() is None:
                print(f"⛔️ ticker 없음: {row['ticker']}")
                continue

            # ✅ price_date + ticker 조합 확인
            cur.execute("""
                SELECT 1 FROM stock_price
                WHERE ticker = %s AND price_date = %s
            """, (row['ticker'], row['price_date']))
            if cur.fetchone() is None:
                print(f"⛔️ stock_price 없음: {row['ticker']}, {row['price_date']}")
                continue

            # ✅ publisher 처리
            cur.execute("""
                INSERT INTO publisher (name)
                VALUES (%s)

                RETURNING publisher_id
            """, (row['publisher_name'],))
            result = cur.fetchone()
            if result:
                publisher_id = result[0]
            else:
                cur.execute("SELECT publisher_id FROM publisher WHERE name = %s", (row['publisher_name'],))
                publisher_id = cur.fetchone()[0]

            # ✅ news 삽입
            cur.execute("""
                INSERT INTO news (
                    ticker, price_date, publisher_id, title, summary, content, url,
                    sentiment, sentiment_score, published_at
                )
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        
                RETURNING news_id
            """, (
                row['ticker'],
                row['price_date'],
                publisher_id,
                row['title'],
                row.get('summary', ''),
                row.get('content', ''),
                row['url'],
                row.get('sentiment', None),
                float(row.get('sentiment_score', 0)),
                row['published_at']
            ))
            result = cur.fetchone()
            if result:
                news_id = result[0]
            else:
                cur.execute("SELECT news_id FROM news WHERE url = %s", (row['url'],))
                news_id = cur.fetchone()[0]

            # ✅ 키워드 처리 및 연결
            keywords = str(row.get('keywords', '')).split(',')
            for word in keywords:
                word = word.strip()
                if not word:
                    continue

                cur.execute("""
                    INSERT INTO keyword (word)
                    VALUES (%s)
             
                    RETURNING keyword_id
                """, (word,))
                result = cur.fetchone()
                if result:
                    keyword_id = result[0]
                else:
                    cur.execute("SELECT keyword_id FROM keyword WHERE word = %s", (word,))
                    keyword_id = cur.fetchone()[0]

                cur.execute("""
                    INSERT INTO news_keyword (news_id, keyword_id)
                    VALUES (%s, %s)
                    ON CONFLICT DO NOTHING
                """, (news_id, keyword_id))

            inserted_count += 1

        except Exception as row_error:
            print(f"[❌ row 오류] {row.get('title', '제목없음')} - {row_error}")
            try:
                if conn and conn.closed == 0:
                    conn.rollback()
            except Exception as rollback_error:
                print(f"[⚠️ rollback 실패] {rollback_error}")
            continue

    conn.commit()
    print(f"✅ 총 {inserted_count}건 삽입 완료.")

except Exception as e:
    print(f"[❌ 전체 오류] {e}")

finally:
    try:
        if cur and not cur.closed:
            cur.close()
        if conn and conn.closed == 0:
            conn.close()
    except Exception as close_error:
        print(f"[⚠️ 종료 중 에러] {close_error}")

[❌ row 오류] HL D&I한라, 이천 '부발역 에피트 에디션' 7월 분양 - 오류:  중복된 키 값이 "unique_publisher_name" 고유 제약 조건을 위반함
DETAIL:  (name)=(fetv) 키가 이미 있습니다.

[❌ row 오류] HL D&I한라, 이천 '부발역 에피트 에디션' 7월 분양 - 오류:  중복된 키 값이 "unique_publisher_name" 고유 제약 조건을 위반함
DETAIL:  (name)=(fetv) 키가 이미 있습니다.

[❌ row 오류] HL D&I한라, 이천 '부발역 에피트 에디션' 7월 분양 - 오류:  중복된 키 값이 "unique_publisher_name" 고유 제약 조건을 위반함
DETAIL:  (name)=(fetv) 키가 이미 있습니다.

[❌ row 오류] HL D&I한라, 이천 '부발역 에피트 에디션' 7월 분양 - 오류:  중복된 키 값이 "unique_publisher_name" 고유 제약 조건을 위반함
DETAIL:  (name)=(fetv) 키가 이미 있습니다.

[❌ row 오류] HL D&I한라, 이천 '부발역 에피트 에디션' 7월 분양 - 오류:  중복된 키 값이 "unique_publisher_name" 고유 제약 조건을 위반함
DETAIL:  (name)=(fetv) 키가 이미 있습니다.

[❌ row 오류] HL D&I한라, 이천 '부발역 에피트 에디션' 7월 분양 - 오류:  중복된 키 값이 "unique_publisher_name" 고유 제약 조건을 위반함
DETAIL:  (name)=(fetv) 키가 이미 있습니다.

[❌ row 오류] HL D&I한라, 이천 '부발역 에피트 에디션' 7월 분양 - 오류:  중복된 키 값이 "unique_publisher_name" 고유 제약 조건을 위반함
DETAIL:  (name)=(fetv) 키가 이미 있습니다.

[❌ row 오류] HL D&I한라, 이천 '부발역 에피트 에디션' 7월 분양 - 오류:  중복된 키 값이 "u

In [1]:
import pandas as pd
import psycopg2
import os
from dotenv import load_dotenv

# 🔐 환경 변수 로드
load_dotenv('stock.env')

# 📦 DB 연결
conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    port=os.getenv("DB_PORT")
)

inserted_count = 0

try:
    cur = conn.cursor()

    # 📥 엑셀 데이터 로드 및 전처리
    df = pd.read_excel(r"C:\Users\wngus\Documents\SK하이닉스_정리본.xlsx")
    df['published_at'] = pd.to_datetime(df['published_at'])
    df['price_date'] = pd.to_datetime(df['price_date'])
    df['ticker'] = df['ticker'].astype(str).str.zfill(6)

    for _, row in df.iterrows():
        try:
            # ✅ ticker 존재 확인
            cur.execute("SELECT ticker FROM ticker WHERE ticker = %s", (row['ticker'],))
            if cur.fetchone() is None:
                print(f"⛔️ ticker 없음: {row['ticker']}")
                continue

            # ✅ stock_price 확인
            cur.execute("""
                SELECT 1 FROM stock_price
                WHERE ticker = %s AND price_date = %s
            """, (row['ticker'], row['price_date']))
            if cur.fetchone() is None:
                print(f"⛔️ stock_price 없음: {row['ticker']}, {row['price_date']}")
                continue

            # ✅ publisher 처리 (중복 안전하게)
            cur.execute("""
                WITH ins AS (
                    INSERT INTO publisher (name)
                    VALUES (%s)
                    ON CONFLICT (name) DO NOTHING
                    RETURNING publisher_id
                )
                SELECT publisher_id FROM ins
                UNION
                SELECT publisher_id FROM publisher WHERE name = %s;
            """, (row['publisher_name'], row['publisher_name']))
            publisher_id = cur.fetchone()[0]

            # ✅ news 삽입
            cur.execute("""
                INSERT INTO news (
                    ticker, price_date, publisher_id, title, summary, content, url,
                    sentiment, sentiment_score, published_at
                )
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                ON CONFLICT (url) DO NOTHING
                RETURNING news_id
            """, (
                row['ticker'],
                row['price_date'],
                publisher_id,
                row['title'],
                row.get('summary', ''),
                row.get('content', ''),
                row['url'],
                row.get('sentiment', None),
                float(row.get('sentiment_score', 0)),
                row['published_at']
            ))

            result = cur.fetchone()
            if result:
                news_id = result[0]
            else:
                cur.execute("SELECT news_id FROM news WHERE url = %s", (row['url'],))
                news_id = cur.fetchone()[0]

            # ✅ 키워드 처리 및 연결
            keywords = str(row.get('keywords', '')).split(',')
            for word in keywords:
                word = word.strip()
                if not word:
                    continue

                cur.execute("""
                    WITH ins AS (
                        INSERT INTO keyword (word)
                        VALUES (%s)
                        ON CONFLICT (word) DO NOTHING
                        RETURNING keyword_id
                    )
                    SELECT keyword_id FROM ins
                    UNION
                    SELECT keyword_id FROM keyword WHERE word = %s;
                """, (word, word))
                keyword_id = cur.fetchone()[0]

                cur.execute("""
                    INSERT INTO news_keyword (news_id, keyword_id)
                    VALUES (%s, %s)
                    ON CONFLICT DO NOTHING
                """, (news_id, keyword_id))

            inserted_count += 1

        except Exception as row_error:
            print(f"[❌ row 오류] {row.get('title', '제목없음')} - {row_error}")
            try:
                if conn and conn.closed == 0:
                    conn.rollback()
            except Exception as rollback_error:
                print(f"[⚠️ rollback 실패] {rollback_error}")
            continue

    conn.commit()
    print(f"✅ 총 {inserted_count}건 삽입 완료.")

except Exception as e:
    print(f"[❌ 전체 오류] {e}")

finally:
    try:
        if cur and not cur.closed:
            cur.close()
        if conn and conn.closed == 0:
            conn.close()
    except Exception as close_error:
        print(f"[⚠️ 종료 중 에러] {close_error}")

✅ 총 860건 삽입 완료.


In [ ]:
import pandas as pd
import psycopg2
import os
from dotenv import load_dotenv

# 🔐 환경 변수 로드
load_dotenv('stock.env')

# 🧠 DB 연결
conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    port=os.getenv("DB_PORT")
)

inserted_count = 0

try:
    cur = conn.cursor()

    # 📥 엑셀 데이터 로드 및 전처리
    df = pd.read_excel(r"C:\Users\wngus\Documents\SK하이닉스_정리본.xlsx")
    df['published_at'] = pd.to_datetime(df['published_at'])
    df['price_date'] = pd.to_datetime(df['price_date'])
    df['ticker'] = df['ticker'].astype(str).str.zfill(6)
    df['url'] = df['url'].astype(str).str.strip()
    df.drop_duplicates(subset='url', inplace=True)

    for _, row in df.iterrows():
        try:
            # ✅ ticker 확인
            cur.execute("SELECT ticker FROM ticker WHERE ticker = %s", (row['ticker'],))
            if cur.fetchone() is None:
                print(f"⛔️ [ticker 없음] {row['ticker']} / {row['title']}")
                continue

            # ✅ stock_price 확인
            cur.execute("""
                SELECT 1 FROM stock_price
                WHERE ticker = %s AND price_date = %s
            """, (row['ticker'], row['price_date']))
            if cur.fetchone() is None:
                print(f"⛔️ [stock_price 없음] {row['ticker']}, {row['price_date']} / {row['title']}")
                continue

            # ✅ publisher 처리
            cur.execute("""
                WITH ins AS (
                    INSERT INTO publisher (name)
                    VALUES (%s)
                    ON CONFLICT (name) DO NOTHING
                    RETURNING publisher_id
                )
                SELECT publisher_id FROM ins
                UNION
                SELECT publisher_id FROM publisher WHERE name = %s;
            """, (row['publisher_name'], row['publisher_name']))
            publisher_id = cur.fetchone()[0]

            # ✅ news 삽입
            print(f"▶ 삽입 시도: {row['title']} / {row['url']}")
            cur.execute("""
                INSERT INTO news (
                    ticker, price_date, publisher_id, title, summary, content, url,
                    sentiment, sentiment_score, published_at
                )
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                ON CONFLICT (url) DO NOTHING
                RETURNING news_id
            """, (
                row['ticker'],
                row['price_date'],
                publisher_id,
                row['title'],
                row.get('summary', ''),
                row.get('content', ''),
                row['url'],
                row.get('sentiment', None),
                float(row.get('sentiment_score', 0)),
                row['published_at']
            ))

            result = cur.fetchone()
            if result:
                news_id = result[0]
                print(f"✅ 삽입 성공: {row['title']}")
                inserted_count += 1
            else:
                print(f"⏩ 중복 URL로 무시됨: {row['title']}")
                cur.execute("SELECT news_id FROM news WHERE url = %s", (row['url'],))
                news_id = cur.fetchone()[0]

            # ✅ 키워드 처리 및 연결
            keywords = str(row.get('keywords', '')).split(',')
            for word in keywords:
                word = word.strip()
                if not word:
                    continue

                cur.execute("""
                    WITH ins AS (
                        INSERT INTO keyword (word)
                        VALUES (%s)
                        ON CONFLICT (word) DO NOTHING
                        RETURNING keyword_id
                    )
                    SELECT keyword_id FROM ins
                    UNION
                    SELECT keyword_id FROM keyword WHERE word = %s;
                """, (word, word))
                keyword_id = cur.fetchone()[0]

                cur.execute("""
                    INSERT INTO news_keyword (news_id, keyword_id)
                    VALUES (%s, %s)
                    ON CONFLICT DO NOTHING
                """, (news_id, keyword_id))

        except Exception as row_error:
            print(f"[❌ row 오류] {row.get('title', '제목없음')} - {row_error}")
            try:
                if conn and conn.closed == 0:
                    conn.rollback()
            except Exception as rollback_error:
                print(f"[⚠️ rollback 실패] {rollback_error}")
            continue

    conn.commit()
    print(f"✅ 실제 삽입된 뉴스: {inserted_count}건")

except Exception as e:
    print(f"[❌ 전체 오류] {e}")

finally:
    try:
        if cur and not cur.closed:
            cur.close()
        if conn and conn.closed == 0:
            conn.close()
    except Exception as close_error:
        print(f"[⚠️ 종료 중 에러] {close_error}")



▶ 삽입 시도: HL D&I한라, 이천 '부발역 에피트 에디션' 7월 분양 / https://www.fetv.co.kr/news/article.html?no=194849
⏩ 중복 URL로 무시됨: HL D&I한라, 이천 '부발역 에피트 에디션' 7월 분양
▶ 삽입 시도: 현대차, 네이버에 시총 밀려…HD현대중공업, 10위권 이탈 / https://n.news.naver.com/mnews/article/008/0005211460?sid=101
⏩ 중복 URL로 무시됨: 현대차, 네이버에 시총 밀려…HD현대중공업, 10위권 이탈
▶ 삽입 시도: 울산에서 AI 데이터 대항해 시대 열린다 / http://www.finomy.com/news/articleView.html?idxno=231861
⏩ 중복 URL로 무시됨: 울산에서 AI 데이터 대항해 시대 열린다
▶ 삽입 시도: "AI 생태계 주권을 잡아라"...삼각전선 형성한 SK·네이버·LG CNS / https://www.ezyeconomy.com/news/articleView.html?idxno=215769
⏩ 중복 URL로 무시됨: "AI 생태계 주권을 잡아라"...삼각전선 형성한 SK·네이버·LG CNS
▶ 삽입 시도: SK하이닉스, 2분기 ‘최대수익 경신’ 예고…반도체 1위 강화 / https://www.ekn.kr/web/view.php?key=20250623023529320
⏩ 중복 URL로 무시됨: SK하이닉스, 2분기 ‘최대수익 경신’ 예고…반도체 1위 강화
▶ 삽입 시도: '반도체 패키징 산업전' 수원서 8월 개최…국내외 기업 참여 / https://n.news.naver.com/mnews/article/421/0008328030?sid=102
⏩ 중복 URL로 무시됨: '반도체 패키징 산업전' 수원서 8월 개최…국내외 기업 참여
▶ 삽입 시도: [오늘증시]코스피, 미 이란 공습 여파에도 3010선 사수…개인 순매수 1조... / http://www.queen.co.kr/news/articleView